In [1]:
# 문서 로딩
documents = [
    "Adapterz is a developer book serving service provided by Startupcode. Adapterz offers textbooks in various fields including programming, artificial intelligence, and data analysis.",
    "Startupcode is a company specializing in developer education. It operates a practice-oriented curriculum and provides textbooks to students through Adapterz.",
    "RAG retrieves external data, inserts it into the LLM input, and then generates an answer. It is used for internal document-based Q&A, customer support chatbots, and more.",
    "Fine-tuning directly modifies model weights to specialize in a domain. It requires retraining whenever data changes, and GPU costs are incurred.",
]

print(f"Number of documents: {len(documents)}")

Number of documents: 4


In [2]:
# 전처리 + 청킹
import re

def preprocess(text):
    """Preprocess: remove unnecessary whitespace"""
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def split_into_chunks(text, chunk_size=120, overlap=30):
    """Split text into fixed-length chunks with overlap"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    return chunks

all_chunks = []
for doc in documents:
    cleaned = preprocess(doc)
    chunks = split_into_chunks(cleaned)
    all_chunks.extend(chunks)

print(f"Number of chunks: {len(all_chunks)}")

for i, chunk in enumerate(all_chunks):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)

Number of chunks: 8

--- Chunk 1 (120 chars) ---
Adapterz is a developer book serving service provided by Startupcode. Adapterz offers textbooks in various fields includ

--- Chunk 2 (90 chars) ---
books in various fields including programming, artificial intelligence, and data analysis.

--- Chunk 3 (120 chars) ---
Startupcode is a company specializing in developer education. It operates a practice-oriented curriculum and provides te

--- Chunk 4 (67 chars) ---
ted curriculum and provides textbooks to students through Adapterz.

--- Chunk 5 (120 chars) ---
RAG retrieves external data, inserts it into the LLM input, and then generates an answer. It is used for internal docume

--- Chunk 6 (80 chars) ---
It is used for internal document-based Q&A, customer support chatbots, and more.

--- Chunk 7 (120 chars) ---
Fine-tuning directly modifies model weights to specialize in a domain. It requires retraining whenever data changes, and

--- Chunk 8 (54 chars) ---
ing whenever data changes, a

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embedding_model.encode(all_chunks)

print(f"벡터 저장 완료")
print(f"  청크 수: {len(chunk_vectors)}") # 8: 청크가 8개 있다는 뜻입니다.
print(f"  벡터 차원: {chunk_vectors.shape[1]}") # 384: 각 청크가 384개의 숫자로 표현되었다는 뜻

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

벡터 저장 완료
  청크 수: 8
  벡터 차원: 384


In [4]:
# 유사 문서 검색(Retrieval)

def consine_similarity(vec_a, vec_b):
    """두 벡터의 코사인 유사도 계산"""
    dot = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a) # 벡터의 크기(len, magnitude) 구하는 함수
    norm_b = np.linalg.norm(vec_b)
    return dot / (norm_a * norm_b)

def search(query, chunk_vectors, all_chunks, embedding_model, top_k=2):
    """질문과 가장 유사한 청크 top_k개를 검색"""
    query_vector = embedding_model.encode(query)
    scores = []
    for i, chunk_vec in enumerate(chunk_vectors):
        sim = consine_similarity(query_vector, chunk_vec)
        scores.append((i, sim, all_chunks[i]))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

question = "What is Adapterz?"
results = search(question, chunk_vectors, all_chunks, embedding_model)

print(f"Question: {question}\n")
for idx, sim, chunk in results:
    print(f"Chunk {idx+1} Similarity: {sim:.4f}")
    print(f"Content: {chunk}")
    print()

Question: What is Adapterz?

Chunk 1 Similarity: 0.7219
Content: Adapterz is a developer book serving service provided by Startupcode. Adapterz offers textbooks in various fields includ

Chunk 4 Similarity: 0.3204
Content: ted curriculum and provides textbooks to students through Adapterz.



In [5]:
# 프롬프트 구성 및 답변 생성 (Augmentation + Generation)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_answer(prompt, tokenizer, model, max_length=200):
    """flan-t5로 텍스트 생성"""
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=max_length)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def build_prompt(question, retrieved_chunks):
    """Build prompt with retrieved chunks"""
    context = "\n".join([chunk for _, _, chunk in retrieved_chunks])
    prompt = (
        f"Answer the question based only on the following documents.\n"
        f"If the answer is not in the documents, say 'The information is not found in the documents.'\n\n"
        f"Documents:\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )
    return prompt

def rag_query(question, chunk_vectors, all_chunks, embedding_model, tokenizer, model, top_k=2):
    """RAG full pipeline: Retrieve -> Augment -> Generate"""
    # Retrieval
    results = search(question, chunk_vectors, all_chunks, embedding_model, top_k)
    # Augmentation
    prompt = build_prompt(question, results)
    # Generation
    answer = generate_answer(prompt, tokenizer, model)
    return answer, results

question = "What is Adapterz?"
answer, results = rag_query(question, chunk_vectors, all_chunks, embedding_model, tokenizer, model)

print(f"Q: {question}")
print(f"A: {answer}")

print(f"\n[Retrieved documents]")
for idx, sim, chunk in results:
    print(f"Chunk {idx+1} (similarity {sim:.4f}: {chunk[:60]}...)")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q: What is Adapterz?
A: a developer book serving service

[Retrieved documents]
Chunk 1 (similarity 0.7219: Adapterz is a developer book serving service provided by Sta...)
Chunk 4 (similarity 0.3204: ted curriculum and provides textbooks to students through Ad...)
